In [2]:
import requests

# Raw URL of the input.txt file on GitHub
url = "https://raw.githubusercontent.com/ishwarraja/SOAI/main/ERAv4/S12/input.txt"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Write the content to a file named 'input.txt'
    with open('input.txt', 'wb') as f:
        f.write(response.content)
    print("input.txt downloaded successfully.")
else:
    print(f"Failed to download input.txt. Status code: {response.status_code}")

input.txt downloaded successfully.


### 1. Install `huggingface_hub`
First, you need to install the `huggingface_hub` library, which provides tools to interact with the Hugging Face Hub.

In [3]:
!pip install huggingface_hub


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


### 2. Log in to Hugging Face
You'll need a Hugging Face account and an access token. You can create a token [here](https://huggingface.co/settings/tokens). Then, log in using your token.

In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

### 3. Save the Model Weights and Vocabulary
Assuming you have successfully run the training code and a model checkpoint (`.pt` file) has been saved, you'll need to save the model's `state_dict`. Additionally, for this character-level GPT, the `DataLoaderLite` defines the vocabulary (`stoi` and `itos` mappings). We should save these to be able to reconstruct the tokenizer when loading the model.

In [4]:
import torch
import json
import os

# --- Assuming the 'main' function has already run and saved a checkpoint ---
# You might need to reload your model here if the current runtime doesn't have it
# For demonstration, let's assume 'model' and 'train_loader' objects are available.
# If not, you would need to instantiate GPTConfig, GPT, and DataLoaderLite first

# Example: if you need to load the model after training finished and runtime was reset
# from your main() function:
# B, T = 4, 1024 # Match the training parameters
# with open('input.txt', 'r') as f: input_text = f.read()
# train_loader = DataLoaderLite(input_text, B=B, T=T)
# config = GPTConfig(vocab_size=train_loader.vocab_size, block_size=T)
# model = GPT(config)
# model_path = 'gpt_124m_final_loss_X.XXXX.pt' # Replace with your actual saved model path
# model.load_state_dict(torch.load(model_path, map_location='cpu'))
# model.eval()

# Make sure 'model' and 'train_loader' are available in the current scope
if 'model' not in locals() or 'train_loader' not in locals():
    print("Model or DataLoader not found in current scope. Please ensure 'main()' has run or objects are re-initialized.")
    # Example re-initialization (uncomment and adjust if needed)
    # with open('input.txt', 'r') as f: input_text = f.read()
    # B, T = 4, 1024
    # train_loader = DataLoaderLite(input_text, B=B, T=T)
    # config = GPTConfig(vocab_size=train_loader.vocab_size, block_size=T)
    # model = GPT(config)
    # model_path = 'gpt_124m_final_loss_2.2142.pt' # Adjust to your actual saved file
    # model.load_state_dict(torch.load(model_path, map_location='cpu'))
    # model.eval()
    # model.data_loader = train_loader # Re-attach for generate function if needed

# Ensure a model checkpoint file exists from previous training
# Assuming 'gpt_124m_final_loss_X.XXXX.pt' was created by the main() function
# Find the latest saved model checkpoint
model_files = [f for f in os.listdir('.') if f.startswith('gpt_124m_final_loss_') and f.endswith('.pt')]
if not model_files:
    print("No model checkpoint found. Please run the training code first.")
    # Exit or handle error
else:
    # Sort to get the latest or specific one if multiple exist
    latest_model_file = sorted(model_files)[-1]
    print(f"Using model checkpoint: {latest_model_file}")

    # Use the current directory to save the model assets
    model_dir = "."

    # Save the model's state_dict
    torch.save(model.state_dict(), os.path.join(model_dir, "pytorch_model.bin"))
    print("Model weights saved to pytorch_model.bin")

    # Save the vocabulary mappings
    with open(os.path.join(model_dir, "vocab.json"), "w") as f:
        json.dump({"stoi": train_loader.stoi, "itos": train_loader.itos, "vocab_size": train_loader.vocab_size}, f)
    print("Vocabulary saved to vocab.json")

    # Save GPTConfig (optional, but good practice)
    with open(os.path.join(model_dir, "config.json"), "w") as f:
        json.dump(model.config.__dict__, f) # Convert dataclass to dict for saving
    print("GPTConfig saved to config.json")


Model or DataLoader not found in current scope. Please ensure 'main()' has run or objects are re-initialized.
No model checkpoint found. Please run the training code first.


In [5]:
!pwd

/Users/isorapad/Documents/Programming/Regular/TESTPYTHON/ERA/v4/s12


### 4. Create a `README.md`
It's good practice to include a `README.md` file that describes your model, its usage, and any relevant details.

In [4]:
import os

model_dir = "." # Explicitly define model_dir for this cell to be the current directory

readme_content = """
---
base_model: GPT-2-like-character-level
tags:
- text-generation
- character-level
- pytorch
---

# Character-level GPT Model

This is a custom character-level GPT model trained on a text dataset (e.g., Shakespeare). It's a minimal implementation designed for educational purposes.

## Model Architecture

The model is a Transformer-based decoder-only architecture, similar to GPT-2, but operating at the character level.

- `block_size`: 1024
- `vocab_size`: Dynamically determined from training data
- `n_layer`: 12
- `n_head`: 12
- `n_embd`: 768

## How to Use

To use this model, you'll need the `pytorch_model.bin` (weights) and `vocab.json` (character mappings).

```python
import torch
import json
from dataclasses import dataclass
import torch.nn as nn
from torch.nn import functional as F
import math

# --- Define your model classes (GPTConfig, CausalSelfAttention, MLP, Block, GPT) here ---
# Copy the relevant classes from your training script.

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

# ... (CausalSelfAttention, MLP, Block, GPT class definitions) ...

class CausalSelfAttention(nn.Module):
    '''A minimal Causal Self-Attention block.'''
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                            .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    '''A minimal Multi-Layer Perceptron block.'''
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    '''A minimal Transformer Block consisting of Attention and MLP.'''
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    '''The full GPT model composed of Blocks.'''
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying
        self.apply(self._init_weights)

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANGPT_SCALE_INIT'):
                std *= module.NANGPT_SCALE_INIT
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        device = idx.device
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"

        pos = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(0)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss


# --- Custom tokenizer based on vocab.json ---
class SimpleCharTokenizer:
    def __init__(self, vocab_file):
        with open(vocab_file, 'r') as f:
            vocab_data = json.load(f)
        self.stoi = vocab_data['stoi']
        self.itos = {int(k): v for k, v in vocab_data['itos'].items()} # keys are string in json
        self.vocab_size = vocab_data['vocab_size']

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, l):
        return ''.join([self.itos[i] for i in l])


# --- Generation function (simplified) ---
def generate_from_hf(model, tokenizer, start_str, max_new_tokens, temperature=1.0, top_k=50, device='cpu'):
    model.eval()
    B, T_model = 1, model.config.block_size # Model's block_size

    start_ids = tokenizer.encode(start_str)
    x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])

    x = x[:, -T_model:] # Truncate if start string is too long for model's block_size

    for _ in range(max_new_tokens):
        # crop context if necessary
        x_cond = x if x.size(1) <= T_model else x[:, -T_model:]

        with torch.no_grad():
            logits, _ = model(x_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        x = torch.cat((x, idx_next), dim=1)

        if tokenizer.stoi.get('\n') is not None and idx_next.item() == tokenizer.stoi.get('\n'):
             break

    return tokenizer.decode(x[0].tolist())



# Example usage:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# # Load config and vocab
# with open('my_gpt_model/config.json', 'r') as f:
#     model_config_dict = json.load(f)
# model_config = GPTConfig(**model_config_dict)
#
# tokenizer = SimpleCharTokenizer('my_gpt_model/vocab.json')
# model = GPT(model_config).to(device)
# model.load_state_dict(torch.load('my_gpt_model/pytorch_model.bin', map_location=device))
#
# prompt = "First Citizen:"
# generated_text = generate_from_hf(model, tokenizer, prompt, max_new_tokens=200, temperature=0.9, device=device)
# print(generated_text)

```

## Files in `.` directory:
- `pytorch_model.bin`: Contains the model's state dictionary (weights).
- `vocab.json`: Contains the character-to-integer (`stoi`) and integer-to-character (`itos`) mappings.
- `config.json`: Contains the model's configuration parameters (`GPTConfig`).

## How to Load and Generate Text

```python
# (Refer to the example usage in the code block above for loading and generating text)
```

**Note**: The model architecture classes (`GPTConfig`, `CausalSelfAttention`, `MLP`, `Block`, `GPT`) and the `generate` function itself are part of the model's definition and would need to be present in your environment when loading the model from Hugging Face. The `README.md` includes these definitions for clarity and ease of use.
"""

# Write the README.md file
with open(os.path.join(model_dir, "README.md"), "w") as f:
    f.write(readme_content)
print("README.md created in current directory.")

README.md created in current directory.


In [9]:
import os

is_colab = False
hf_token_colab_secrets = None

try:
    from google.colab import userdata
    is_colab = True
    hf_token_colab_secrets = userdata.get('HF_TOKEN')
    if hf_token_colab_secrets:
        print(f"Detected Google Colab. Token from Colab secrets (first 5 chars): {hf_token_colab_secrets[:5]}...")
    else:
        print("Detected Google Colab, but HF_TOKEN not found in Colab secrets.")
except ImportError:
    print("Not running in Google Colab (or 'google.colab' not installed).")

# Try to get the token from environment variables (works in Colab and other environments like VS Code)
hf_token_env = os.getenv('HF_TOKEN')
if hf_token_env:
    print(f"Token from environment variable (first 5 chars): {hf_token_env[:5]}...")
else:
    print("HF_TOKEN environment variable not set.")

token_to_use = None
if is_colab and hf_token_colab_secrets:
    token_to_use = hf_token_colab_secrets
    print("Prioritizing Colab secret token.")
elif hf_token_env:
    token_to_use = hf_token_env
    print("Prioritizing environment variable token.")
else:
    print("No Hugging Face token found from Colab secrets or environment variables.")

# Now 'token_to_use' holds the token if one was found
if token_to_use:
    print(f"Final token selected (first 5 chars): {token_to_use[:5]}...")
else:
    print("No Hugging Face token available for use.")

# Example of how to use it:
# from huggingface_hub import HfApi
# if token_to_use:
#     api = HfApi(token=token_to_use)
#     # Further operations with api
# else:
#     print("Cannot initialize HfApi without a token.")

Not running in Google Colab (or 'google.colab' not installed).
Token from environment variable (first 5 chars): hf_JO...
Prioritizing environment variable token.
Final token selected (first 5 chars): hf_JO...


In [ ]:
# from google.colab import userdata
# import os
# from huggingface_hub import HfApi

# # Retrieve your Hugging Face token from Colab Secrets
# hf_token_direct = userdata.get('HF_TOKEN')

# # It's good practice to ensure the token is not None before using it
# if hf_token_direct is None:
#     print("❌ HF_TOKEN not found in Colab Secrets. Please make sure it's set up correctly.")
# else:
#     print("Token accessed directly (first 5 chars):", hf_token_direct[:5] + "...")

#     # Verify connection using Hugging Face API
#     try:
#         api = HfApi(token=hf_token_direct)
#         user_info = api.whoami()  # Check if token works
#         print("✅ Tested Connection Successfully working.")
#         print(f"Logged in as: {user_info['name']}")
#     except Exception as e:
#         print("❌ Connection test failed. Please check your token.")
#         print("Error:", str(e))

Token accessed directly (first 5 chars): hf_JO...
✅ Tested Connection Successfully working.
Logged in as: ishwarraja


In [12]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Configuration for 124M Model ---
@dataclass
class GPTConfig:
    block_size: int = 1024  # Context length (T)
    vocab_size: int = 50257 # Will be updated dynamically based on input.txt
    n_layer: int = 12       # L (Layers) - Set for GPT-2 Small (124M)
    n_head: int = 12        # H (Heads) - Set for GPT-2 Small (124M)
    n_embd: int = 768       # D (Embedding dimension) - Set for GPT-2 Small (124M)

# --- Transformer Components (as provided in initial file snippet) ---

class CausalSelfAttention(nn.Module):
    """A minimal Causal Self-Attention block."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # Scale init to 1.0/sqrt(2L) for residual connection stability
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Causal mask (tril) is registered as buffer
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                            .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Causal self-attention; (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)

        y = att @ v # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs

        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    """A minimal Multi-Layer Perceptron block."""
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        # Scale init to 1.0/sqrt(2L) for residual connection stability
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    """A minimal Transformer Block consisting of Attention and MLP."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    """The full GPT model composed of Blocks."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying

        self.apply(self._init_weights)
        print(f"Number of parameters: {self.get_num_params()/1e6:.2f} Million")

    def get_num_params(self, non_embedding=True):
        """Return the number of parameters in the model."""
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        """Custom weight initialization."""
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANGPT_SCALE_INIT'):
                std *= module.NANGPT_SCALE_INIT
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        device = idx.device
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"

        pos = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(0) # shape (1, T)

        tok_emb = self.transformer.wte(idx) # token embeddings (B, T, D)
        pos_emb = self.transformer.wpe(pos) # position embeddings (1, T, D)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Flatten B, T, V to (B*T), V for cross-entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss

# --- Data Loading and Tokenization (Character-level for Simplicity) ---
class DataLoaderLite:
    def __init__(self, input_text, B, T):
        self.B = B # batch size
        self.T = T # context length

        # Character-level tokenizer
        chars = sorted(list(set(input_text)))
        self.vocab_size = len(chars)
        self.stoi = {ch:i for i,ch in enumerate(chars)}
        self.itos = {i:ch for i,ch in enumerate(chars)}
        self.encode = lambda s: [self.stoi[c] for c in s]
        self.decode = lambda l: ''.join([self.itos[i] for i in l])

        # Convert text to tokens
        self.tokens = torch.tensor(self.encode(input_text), dtype=torch.long)
        print(f"Loaded data with {len(self.tokens)} tokens, vocabulary size: {self.vocab_size}")

        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]

        if buf.size(0) < B * T + 1:
            # Wrap around when end of data is reached
            remainder = B * T + 1 - buf.size(0)
            buf = torch.cat((buf, self.tokens[:remainder]))
            self.current_position = remainder # Restart from the beginning
        else:
            self.current_position += B * T

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        return x, y

# --- Main Execution and Training Loop ---

def generate(model, start_str, max_new_tokens, temperature=1.0, top_k=50):
    """Generates new text given a starting string."""
    model.eval()
    B, T = 1, model.config.block_size

    # Encode the starting string
    start_ids = model.data_loader.encode(start_str)
    x = (torch.tensor(start_ids, dtype=torch.long, device=model.lm_head.weight.device)[None, ...])

    # Truncate if the start string is too long
    x = x[:, -T:]

    for _ in range(max_new_tokens):
        # crop context if necessary
        x_cond = x if x.size(1) <= T else x[:, -T:]

        with torch.no_grad():
            logits, _ = model(x_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally apply top_k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # sample from the distribution
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # append sampled index to the running sequence
        x = torch.cat((x, idx_next), dim=1)

        # stop if we predict the newline token (often a good proxy for end of generation)
        if idx_next.item() == model.data_loader.stoi.get('\n'):
             break

    return model.data_loader.decode(x[0].tolist())

def main():
    # 1. Setup Environment and Load Data

    # Check for GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Set seeds for reproducibility
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)

    try:
        # NOTE: Assumes input.txt is available in the execution environment
        with open('input.txt', 'r') as f:
            input_text = f.read()
    except FileNotFoundError:
        print("ERROR: 'input.txt' not found. Please ensure the file is downloaded to the execution directory.")
        return

    # 2. Initialize Data Loader and Model
    B, T = 4, 1024 # Batch size 4, Context length 1024 (Adjust B down if out-of-memory on Colab T4)
    train_loader = DataLoaderLite(input_text, B=B, T=T)

    # Update config with the dynamically determined vocab size
    config = GPTConfig(vocab_size=train_loader.vocab_size, block_size=T)
    model = GPT(config).to(device)
    model.data_loader = train_loader # Attach data loader for easy access in generate function

    # 3. Training Setup
    max_steps = 10000  # A high number of steps is necessary for the ambitious loss target (< 0.1)
    eval_interval = 100
    log_interval = 10

    # optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, betas=(0.9, 0.95), weight_decay=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.95), weight_decay=0.1)

    # 4. Training Loop
    start_time = time.time()

    for step in range(max_steps):
        # Set model to training mode
        model.train()

        # Fetch batch and move to device
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)

        # Forward pass and backpropagation
        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()

        # Simple gradient clipping to stabilize training
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        # Logging and Evaluation
        current_loss = loss.item()

        if step % log_interval == 0:
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps} | Loss: {current_loss:.6f} | Time: {elapsed_time:.2f}s")

            # Check for early stopping condition (Loss < 0.099999)
            if current_loss < 0.099999:
                print("\n" + "="*50)
                print(f"SUCCESS: Loss target reached at Step {step+1}!")
                print("="*50 + "\n")

                # Final save and sample generation
                torch.save(model.state_dict(), f'gpt_124m_final_loss_{current_loss:.4f}.pt')
                break

        if step % eval_interval == 0 and step > 0:
            print("-" * 50)
            print("--- Generating Sample Output ---")

            # Generate sample text
            prompt = "First Citizen:"
            generated_text = generate(model, prompt, max_new_tokens=200, temperature=0.9)

            print(f"\n[Prompt]: {prompt}\n")
            print(f"[Generated Text]:\n{generated_text}")
            print("-" * 50)

            # Save checkpoint
            torch.save(model.state_dict(), f'checkpoint_step_{step}.pt')

    print("\nTraining complete.")
    if current_loss >= 0.099999:
        print(f"Note: Final loss of {current_loss:.6f} did not meet the target of < 0.099999. Try running for more steps or adjusting hyperparameters.")

if __name__ == '__main__':
    main()

Using device: cpu
Loaded data with 1115394 tokens, vocabulary size: 65
Number of parameters: 85.11 Million
Step 1/10000 | Loss: 4.401192 | Time: 4.61s
Step 11/10000 | Loss: 3.372537 | Time: 40.10s
Step 21/10000 | Loss: 3.020364 | Time: 75.93s
Step 31/10000 | Loss: 2.800066 | Time: 110.05s
Step 41/10000 | Loss: 2.791023 | Time: 144.84s
Step 51/10000 | Loss: 2.663822 | Time: 179.63s
Step 61/10000 | Loss: 2.513137 | Time: 214.93s
Step 71/10000 | Loss: 2.535732 | Time: 249.98s
Step 81/10000 | Loss: 2.608598 | Time: 287.70s
Step 91/10000 | Loss: 2.572037 | Time: 323.53s
Step 101/10000 | Loss: 2.475118 | Time: 358.06s
--------------------------------------------------
--- Generating Sample Output ---

[Prompt]: First Citizen:

[Generated Text]:
First Citizen: re d me teres keiteand or bay ps, he wilimat wond to fe hinein trd linofor n his thaf thathico shee,

--------------------------------------------------
Step 111/10000 | Loss: 2.530987 | Time: 396.64s
Step 121/10000 | Loss: 2.520117 | T

In [14]:
import os

is_colab = False
hf_token_colab_secrets = None

try:
    from google.colab import userdata
    is_colab = True
    hf_token_colab_secrets = userdata.get('HF_TOKEN')
    if hf_token_colab_secrets:
        print(f"Detected Google Colab. Token from Colab secrets (first 5 chars): {hf_token_colab_secrets[:5]}...")
    else:
        print("Detected Google Colab, but HF_TOKEN not found in Colab secrets.")
except ImportError:
    print("Not running in Google Colab (or 'google.colab' not installed).")

# Try to get the token from environment variables (works in Colab and other environments like VS Code)
hf_token_env = os.getenv('HF_TOKEN')
if hf_token_env:
    print(f"Token from environment variable (first 5 chars): {hf_token_env[:5]}...")
else:
    print("HF_TOKEN environment variable not set.")

token_to_use = None
if is_colab and hf_token_colab_secrets:
    token_to_use = hf_token_colab_secrets
    print("Prioritizing Colab secret token.")
elif hf_token_env:
    token_to_use = hf_token_env
    print("Prioritizing environment variable token.")
else:
    print("No Hugging Face token found from Colab secrets or environment variables.")

# Now 'token_to_use' holds the token if one was found
if token_to_use:
    print(f"Final token selected (first 5 chars): {token_to_use[:5]}...")
else:
    print("No Hugging Face token available for use.")

# Example of how to use it:
# from huggingface_hub import HfApi
# if token_to_use:
#     api = HfApi(token=token_to_use)
#     # Further operations with api
# else:
#     print("Cannot initialize HfApi without a token.")

Not running in Google Colab (or 'google.colab' not installed).
Token from environment variable (first 5 chars): hf_JO...
Prioritizing environment variable token.
Final token selected (first 5 chars): hf_JO...


In [ ]:
from huggingface_hub import create_repo, upload_folder
import os # Import os to potentially set the token as an environment variable (recommended)

# ----------------------------------------------------------------------------------
# 🔑 IMPORTANT CHANGE:
# In Colab, the token was read from a secret. For local VS Code, you should
# use an environment variable or directly paste your token (less secure but simpler).
#
# Replace 'YOUR_ACTUAL_HUGGING_FACE_TOKEN' with your token string.
# A better practice is to load it from an environment variable (see below).
# ----------------------------------------------------------------------------------

# OPTION 1 (Recommended: Environment Variable)
# Set the environment variable HUGGINGFACE_TOKEN in your terminal *before* running:
# export HUGGINGFACE_TOKEN="YOUR_ACTUAL_TOKEN"
# token = os.environ.get("HUGGINGFACE_TOKEN")

# OPTION 2 (Direct Assignment - Use with caution for security)
token = os.getenv('VSCode_Upload')
# token = "HF_Token"

# --- The rest of the code remains largely the same ---

# Define the repository ID
repo_id = "ishwarraja/s12" 

# Create a repository if it doesn't exist, passing the token for authentication
create_repo(repo_id, exist_ok=True, token=token)
print(f"Repository '{repo_id}' ensured to exist.")

# Upload the model directory
# **Ensure your model files are in the same directory as this script.**
upload_folder(
    repo_id=repo_id,
    folder_path=".", # Stays as "." if the files are in the current directory
    commit_message="Upload initial character-level GPT model",
    ignore_patterns=["*.ptc"],
    token=token
)

print(f"Model successfully uploaded to https://huggingface.co/{repo_id}")

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6928335f-294105c1331839aa02531874;4df7bcd3-f130-451e-8a22-b7fe0db01a14)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

User Access Token "Token1" is expired

FOR GOOGLE Collab

## Summary:

### Data Analysis Key Findings
*   A `TypeError` occurring when `hf_token_env` was `None` was successfully resolved by implementing a conditional check, allowing safe printing of the token status.
*   The Hugging Face token obtained directly via `userdata.get()` (`hf_token_direct`) was consistently valid, whereas the token accessed as an environment variable via `os.getenv()` (`hf_token_env`) consistently returned `None`.
*   Despite `hf_token_env` being `None`, the Hugging Face API connection was successfully established and verified, as the `HfApi` object was initialized using the valid `hf_token_direct`.
*   A previous `HfHubHTTPError` (401 Client Error: Unauthorized) during model upload was resolved by explicitly defining the repository ID as "ishwarraja/my-character-gpt," using the successfully retrieved `hf_token_direct` for authentication, and correcting the upload path from 's12' to '.'
*   The character-level GPT model was successfully uploaded to the Hugging Face Hub under the repository `ishwarraja/my-character-gpt`.

### Insights or Next Steps
*   Always implement robust checks for `None` values when processing user data or environment variables to prevent runtime errors like `TypeError`.
*   To ensure reliable authentication for services like Hugging Face, prioritize direct retrieval methods (e.g., `userdata.get()`) if environment variable access is inconsistent.
